# Sentiment Analysis on Energy - Ireland

In [3]:
# Import Libraries
import requests
import json
import os
import pandas as pd

from dotenv import load_dotenv

### Load Credentials

In [6]:
load_dotenv()

CLIENT_ID = os.getenv('CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET')
USERNAME = os.getenv('REDDIT_USERNAME')
PASSWORD = os.getenv('REDDIT_PASSWORD')
USER_AGENT= os.getenv('REDDIT_USERAGENT')

if CLIENT_ID and CLIENT_SECRET and USERNAME and PASSWORD:
    print('Credentials Loaded!')
else:
    print('Error loading credentials!')

Credentials Loaded!


In [8]:
data = {
    'grant_type': 'password',
    'username': USERNAME,
    'password': PASSWORD
}

auth = requests.auth.HTTPBasicAuth(CLIENT_ID, CLIENT_SECRET)


headers = {'User-Agent': USER_AGENT}

# Get access token
res = requests.post('https://www.reddit.com/api/v1/access_token',
                    auth=auth, data=data, headers=headers)

if res.status_code != 200:
    print("Token request failed:", res.text)
    exit()

token = res.json()['access_token']

### Data Gathering

In [11]:
# Sources:

POST_ID=['1b3qqxh/electricity_bill/',
         '1ebph9r/electricity_providers/',
         '1dndzwv/best_energy_provider_to_go_with/',
         '1cuw7de/insane_electricity_usage/',
         '1gk49ps/blow_for_homeowners_with_solar_panels_as_electric/',
         '1h1s73w/irelands_data_centres_turning_to_fossil_fuels/',
         '16nkwbq/currently_71_of_irelands_electricity_from_wind/',
         '12yqhv8/can_someone_eli5_why_our_electricity_prices_are/',
        '1j07rjd/sse_airtricity_to_hike_electricity_gas_prices/',
        '1exjvob/very_high_electricity_bill/']

In [13]:
import re

headers = {
    'Authorization': f'bearer {token}',
    'User-Agent': USER_AGENT
}

sentiment_arr = []
append_f = lambda x, y: sentiment_arr.append({'source':x,'comment':str(y) })

re_pattern=r'"(?:selftext|body)"\s*:\s*"((?:\\.|[^"\\])*)"'

for _id in POST_ID:
    try:
        url = f'https://oauth.reddit.com/comments/{_id}'
        print(f'getting data from {url}')
        
        response = requests.get(url, headers=headers)
        
        if response.status_code != 200:
            print("Failed to fetch comments:", response.text)
            exit()
        
        matches = re.findall(re_pattern, json.dumps(response.json()))  
        
        for match in matches:
          append_f(_id, match)
    except Exception as e:
        print(f'Error retrieving data from {_id}, error details: {e}')

print('Processes completed!')

getting data from https://oauth.reddit.com/comments/1b3qqxh/electricity_bill/
getting data from https://oauth.reddit.com/comments/1ebph9r/electricity_providers/
getting data from https://oauth.reddit.com/comments/1dndzwv/best_energy_provider_to_go_with/
getting data from https://oauth.reddit.com/comments/1cuw7de/insane_electricity_usage/
getting data from https://oauth.reddit.com/comments/1gk49ps/blow_for_homeowners_with_solar_panels_as_electric/
getting data from https://oauth.reddit.com/comments/1h1s73w/irelands_data_centres_turning_to_fossil_fuels/
getting data from https://oauth.reddit.com/comments/16nkwbq/currently_71_of_irelands_electricity_from_wind/
getting data from https://oauth.reddit.com/comments/12yqhv8/can_someone_eli5_why_our_electricity_prices_are/
getting data from https://oauth.reddit.com/comments/1j07rjd/sse_airtricity_to_hike_electricity_gas_prices/
getting data from https://oauth.reddit.com/comments/1exjvob/very_high_electricity_bill/
Processes completed!


In [14]:
sentiments_df = pd.DataFrame(sentiment_arr)
sentiments_df

,source,comment
0,1b3qqxh/electricity_bill/,"So my dear folks, got electricity bill mid dec..."
1,1b3qqxh/electricity_bill/,Do the readings on the bill actually match the...
2,1b3qqxh/electricity_bill/,When did this happen? If you got a refund of o...
3,1b3qqxh/electricity_bill/,I spoke to them yesterday and the way yhey spo...
4,1b3qqxh/electricity_bill/,I'd say /u/Dan_92159 is probably on the money ...
...,...,...
1044,1exjvob/very_high_electricity_bill/,"No, but you can pay the bill as soon as you ge..."
1045,1exjvob/very_high_electricity_bill/,I was thinking if I signed up for direct debit...
1046,1exjvob/very_high_electricity_bill/,Electric Ireland just told us on phone pay off...
1047,1exjvob/very_high_electricity_bill/,My unit is the same because I refuse to go dir...


### Data Cleaning

In [60]:
def custom_preprocessor(text):
    #Remove URLS
    text = re.sub(r"http\S+|www\S+", "", text)
    # Remove special characters
    text = re.sub(r"[^\w\s]", "", text)
    # Normalize white spaces
    text = re.sub(r"\s+", " ", text).strip()
    text=text.lower()
    return text

In [62]:
# Apply custom cleaning to each comment
cleaned_comments = sentiments_df['comment'].apply(custom_preprocessor)
X = vectorizer.fit_transform(cleaned_comments)
feature_names = vectorizer.get_feature_names_out()
X_df = pd.DataFrame(X.toarray(), columns=feature_names)
sentiments_df_with_tokens = sentiments_df.join(X_df)

In [64]:
sentiments_df_with_tokens

,source,comment,actually,ago,average,battery,best,better,big,bills,...,wind,winter,wont,work,world,yeah,year,years,yes,youre
0,1b3qqxh/electricity_bill/,"So my dear folks, got electricity bill mid dec...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1b3qqxh/electricity_bill/,Do the readings on the bill actually match the...,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1b3qqxh/electricity_bill/,When did this happen? If you got a refund of o...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1b3qqxh/electricity_bill/,I spoke to them yesterday and the way yhey spo...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,1b3qqxh/electricity_bill/,I'd say /u/Dan_92159 is probably on the money ...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044,1exjvob/very_high_electricity_bill/,"No, but you can pay the bill as soon as you ge...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1045,1exjvob/very_high_electricity_bill/,I was thinking if I signed up for direct debit...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1046,1exjvob/very_high_electricity_bill/,Electric Ireland just told us on phone pay off...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1047,1exjvob/very_high_electricity_bill/,My unit is the same because I refuse to go dir...,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


## VaderSentiment

In [66]:
sentiments_df['cleaned']=sentiments_df.comment.apply(custom_preprocessor)
sentiments_df

,source,comment,cleaned
0,1b3qqxh/electricity_bill/,"So my dear folks, got electricity bill mid dec...",so my dear folks got electricity bill mid dec ...
1,1b3qqxh/electricity_bill/,Do the readings on the bill actually match the...,do the readings on the bill actually match the...
2,1b3qqxh/electricity_bill/,When did this happen? If you got a refund of o...,when did this happen if you got a refund of ov...
3,1b3qqxh/electricity_bill/,I spoke to them yesterday and the way yhey spo...,i spoke to them yesterday and the way yhey spo...
4,1b3qqxh/electricity_bill/,I'd say /u/Dan_92159 is probably on the money ...,id say udan_92159 is probably on the money and...
...,...,...,...
1044,1exjvob/very_high_electricity_bill/,"No, but you can pay the bill as soon as you ge...",no but you can pay the bill as soon as you get...
1045,1exjvob/very_high_electricity_bill/,I was thinking if I signed up for direct debit...,i was thinking if i signed up for direct debit...
1046,1exjvob/very_high_electricity_bill/,Electric Ireland just told us on phone pay off...,electric ireland just told us on phone pay off...
1047,1exjvob/very_high_electricity_bill/,My unit is the same because I refuse to go dir...,my unit is the same because i refuse to go dir...
